# Simple RAG Training Pipeline

This notebook provides a streamlined 4-step process to generate synthetic QA data and launch a training job:

1. **Point to dataset** - Chunk and upload documents
2. **Create synthetic QA** - Generate question-answer pairs
3. **Create env** - Bundle and upload search environment
4. **Run training job** - Launch the training

## Setup

Install dependencies and configure API credentials.

In [ ]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgft-io/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev]

In [ ]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

In [ ]:
# Configuration
API_KEY = "your-api-key"
BASE_URL = "https://app.cgft.io"

# Dataset configuration
DOCS_PATH = "./samples/posthog"
CORPUS_NAME = "my-docs"

# QA generation configuration
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

NUM_SINGLE_HOP = 40
NUM_MULTI_HOP = 5

## Step 1: Point to Dataset

Chunk your documents and upload to corpus API in one line.

In [ ]:
from synthetic_data_prep.corpus import prepare_corpus

corpus, collection, corpus_client = prepare_corpus(
    docs_path=DOCS_PATH,
    corpus_name=CORPUS_NAME,
    api_key=API_KEY,
    base_url=BASE_URL,
)

## Step 2: Create Synthetic QA

Generate synthetic question-answer pairs from your corpus.

In [ ]:
from synthetic_data_prep.qa_generation import generate_dataset

dataset = generate_dataset(
    corpus=corpus,
    collection=collection,
    corpus_client=corpus_client,
    api_key=API_KEY,
    corpus_description=CORPUS_DESCRIPTION,
    example_queries=EXAMPLE_QUERIES,
    num_single_hop=NUM_SINGLE_HOP,
    num_multi_hop=NUM_MULTI_HOP,
)

## Step 3: Upload Dataset & Create Environment

Upload the dataset and bundle the search environment for training.

In [ ]:
from synthetic_data_prep.trainer import upload_dataset, upload_env
from synthetic_data_prep.envs import SearchEnv

# Upload dataset
dataset_blob_path = upload_dataset(
    dataset=dataset,
    corpus=corpus,
    api_key=API_KEY,
    base_url=BASE_URL,
)

# Upload environment
env_blob_path, env_args_blob_path = upload_env(
    env_class=SearchEnv,
    constructor_args={
        "api_key": API_KEY,
        "corpus_id": corpus.id,
        "base_url": BASE_URL,
        "dataset_path": f"~/user-data/{dataset_blob_path}",
    },
    api_key=API_KEY,
    base_url=BASE_URL,
)

## Step 4: Run Training Job

Launch the training job with your dataset and environment.

In [ ]:
from synthetic_data_prep.trainer import launch_job

job = launch_job(
    dataset_blob_path=dataset_blob_path,
    env_blob_path=env_blob_path,
    api_key=API_KEY,
    base_url=BASE_URL,
)

## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:
- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse

## Complete Example

Here's what the complete flow looks like:

```python
from synthetic_data_prep.corpus import prepare_corpus
from synthetic_data_prep.qa_generation import generate_dataset
from synthetic_data_prep.trainer import upload_dataset, upload_env, launch_job
from synthetic_data_prep.envs import SearchEnv

# 1. Point to dataset
corpus, collection, corpus_client = prepare_corpus(
    docs_path="./samples/posthog",
    corpus_name="my-docs",
    api_key=API_KEY,
)

# 2. Create synthetic QA
dataset = generate_dataset(
    corpus=corpus,
    collection=collection,
    corpus_client=corpus_client,
    api_key=API_KEY,
    corpus_description="Posthog documentation",
    example_queries=["how to feature flag", "setup reverse proxy"],
    num_single_hop=40,
    num_multi_hop=5,
)

# 3. Upload dataset and create env
dataset_path = upload_dataset(dataset, corpus, API_KEY)
env_path, _ = upload_env(
    SearchEnv,
    {"api_key": API_KEY, "corpus_id": corpus.id, "base_url": BASE_URL, "dataset_path": f"~/user-data/{dataset_path}"},
    API_KEY,
)

# 4. Run training job
job = launch_job(dataset_path, env_path, API_KEY)
```